# Champion - WoE Logistic Regression

This notebook builds and evaluates a WoE logistic regression model for 12-month default prediction. It reads the final logistic-ready WoE dataset from `data/final/` and the selected-feature metadata from `data/final/` and `data/logs/`.

In the current run, `SPLIT_STRATEGY = "out_of_time"`. The validation design is therefore hybrid:

- **Final test:** 2021 observations are kept as an independent out-of-time test set.
- **Internal validation:** the 2015-2020 development period is split into model train and validation using stratified company-level sampling.
- **Business interpretation:** the final test checks whether a model developed on earlier years still works in the future period. The internal validation split checks model behavior on companies not used during fitting.

Main workflow:
- load the final WoE modeling dataset with existing split labels
- keep 2021 as the upstream out-of-time test set
- create an internal train / validation split inside the 2015-2020 development period
- fit logistic regression on the internal train set
- check statistical significance and coefficient signs
- choose a classification threshold on validation
- report final performance on the out-of-time test set
- convert model coefficients into a simple scorecard



In [1]:
from __future__ import annotations

import json
from math import log
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_fscore_support, roc_auc_score

FINAL_DIR = Path("data/final")
LOG_DIR = Path("data/logs")

SPLIT_STRATEGY = "out_of_time"

DATA_PATHS = {
    "out_of_sample": FINAL_DIR / "logit_selected_woe_dataset_out_of_sample.csv",
    "out_of_time": FINAL_DIR / "logit_selected_woe_dataset_out_of_time.csv",
}

DATA_PATH = DATA_PATHS[SPLIT_STRATEGY]
FEATURES_PATH = FINAL_DIR / "selected_features_for_logit.json"
FEATURE_TABLE_PATH = LOG_DIR / "selected_features_for_logit.csv"

TARGET_COL = "default"
ID_COL = "ID"
DATE_COL = "obs_date"

SPLIT_COL = "data_split"
RANDOM_STATE = 42
VALIDATION_SIZE = 0.2


## 1. Load WoE modeling data

The notebook loads the final logistic-ready WoE dataset selected by `SPLIT_STRATEGY`. In the current run this is `data/final/logit_selected_woe_dataset_out_of_time.csv`. The selected feature list is loaded from `data/final/selected_features_for_logit.json`, and the feature-name table is loaded from `data/logs/selected_features_for_logit.csv`. The target variable is `default`.


In [2]:
features_payload = json.loads(FEATURES_PATH.read_text())
selected_features = features_payload["selected_features"]
upstream_strategy = features_payload.get("settings", {}).get("modeling_split_strategy")
exported_logit_datasets = features_payload.get("exported_logit_datasets", {})
expected_filename = exported_logit_datasets.get(SPLIT_STRATEGY)
feature_names = pd.read_csv(FEATURE_TABLE_PATH).set_index("feature")["feature_name"].to_dict()

df = pd.read_csv(DATA_PATH, sep=";", low_memory=False)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")

if SPLIT_COL not in df.columns:
    raise KeyError(
        f"Column '{SPLIT_COL}' not found. Run feature_engineering.ipynb first "
        "to create the train / test split."
    )

model_cols = [ID_COL, DATE_COL, TARGET_COL, SPLIT_COL, *selected_features]
model_df = df[model_cols].dropna(subset=[TARGET_COL, *selected_features]).copy()
model_df[TARGET_COL] = model_df[TARGET_COL].astype(int)
model_df[SPLIT_COL] = model_df[SPLIT_COL].astype("string")

expected_splits = {"train", "test"}
actual_splits = set(model_df[SPLIT_COL].dropna().unique())
if actual_splits != expected_splits:
    raise ValueError(f"Expected splits {expected_splits}, found {actual_splits}.")

if expected_filename is not None and DATA_PATH.name != expected_filename:
    raise ValueError(
        f"JSON expects {expected_filename} for {SPLIT_STRATEGY}, but notebook points to {DATA_PATH.name}."
    )

X = model_df[selected_features]
y = model_df[TARGET_COL]

print(f"Selected final dataset strategy: {SPLIT_STRATEGY}")
print(f"Dataset path: {DATA_PATH}")
if upstream_strategy is not None:
    print(f"Upstream strategy recorded in JSON: {upstream_strategy}")

print(f"Rows used: {len(model_df):,}")
print(f"Selected WoE features: {len(selected_features)}")
print(f"Default rate: {y.mean() * 100:.4f}%")

display(model_df.head())


Selected final dataset strategy: out_of_time
Dataset path: data/final/logit_selected_woe_dataset_out_of_time.csv
Upstream strategy recorded in JSON: out_of_time
Rows used: 125,758
Selected WoE features: 9
Default rate: 6.9570%


,ID,obs_date,default,data_split,Var_02,Var_28,Var_04,Var_15,Var_35,Var_23,Var_33,Var_06_missing_flag,Var_07
0,1574,2015-12-31,0,train,5.298025,-0.349729,0.230771,-0.038128,-0.167346,0.056733,0.042769,0.07307,-0.214233
1,1574,2016-12-31,0,train,5.298025,-0.349729,-0.221124,-0.038128,-0.167346,0.056733,0.042769,0.07307,-0.214233
2,1574,2017-12-31,0,train,5.298025,-0.280456,0.289641,-0.038128,-0.379115,-0.461309,0.042769,0.07307,-0.308367
3,1574,2018-12-31,0,train,5.298025,-0.349729,-0.386581,-0.038128,-0.379115,-0.461309,0.042769,0.07307,-0.214233
4,1574,2019-12-31,0,train,5.298025,-0.431094,0.289641,-0.038128,-0.379115,0.056733,0.042769,0.07307,-0.214233


## 2. Create internal train / validation split

The notebook keeps the upstream split that comes with the chosen final dataset. In the current run, `out_of_time` is selected, so 2021 stays as the final test period. Inside the 2015-2020 development period, the notebook creates a validation set with stratified company-level sampling.

Usage of each split:
- **model train:** fit the logistic regression model
- **validation:** choose the classification threshold and check model behavior before final testing
- **final test:** independent out-of-time performance check on 2021

This means the project does not use a purely company-based final test. Instead, it uses an out-of-time final test, which is more appropriate for checking future credit-risk performance. The company-level split is used only inside the development sample to make the validation check more conservative.


In [3]:
development_mask = model_df[SPLIT_COL].eq("train")
test_mask = model_df[SPLIT_COL].eq("test")

def stratified_company_train_validation_ids(
    company_target: pd.Series,
    validation_size: float = VALIDATION_SIZE,
    seed: int = RANDOM_STATE,
) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    labels = company_target.astype(int)

    if labels.nunique() < 2:
        ids = np.array(labels.index.to_numpy(), copy=True)
        rng.shuffle(ids)
        n_valid = max(1, int(np.floor(len(ids) * validation_size)))
        return ids[n_valid:], ids[:n_valid]

    train_parts = []
    valid_parts = []

    for _, label_ids in labels.groupby(labels):
        ids = np.array(label_ids.index.to_numpy(), copy=True)
        rng.shuffle(ids)
        n_valid = max(1, int(np.floor(len(ids) * validation_size)))
        n_valid = min(n_valid, len(ids) - 1)
        valid_parts.append(ids[:n_valid])
        train_parts.append(ids[n_valid:])

    return np.concatenate(train_parts), np.concatenate(valid_parts)

development_company_target = model_df.loc[development_mask].groupby(ID_COL)[TARGET_COL].max()
train_company_ids, valid_company_ids = stratified_company_train_validation_ids(
    development_company_target,
)
train_company_ids = set(train_company_ids.tolist())
valid_company_ids = set(valid_company_ids.tolist())

train_idx = model_df.index[development_mask & model_df[ID_COL].isin(train_company_ids)]
valid_idx = model_df.index[development_mask & model_df[ID_COL].isin(valid_company_ids)]
test_idx = model_df.index[test_mask]

X_train = X.loc[train_idx].copy()
X_valid = X.loc[valid_idx].copy()
X_test = X.loc[test_idx].copy()
y_train = y.loc[train_idx].copy()
y_valid = y.loc[valid_idx].copy()
y_test = y.loc[test_idx].copy()

test_ids = set(model_df.loc[test_idx, ID_COL])
train_valid_overlap = train_company_ids.intersection(valid_company_ids)
train_test_overlap = train_company_ids.intersection(test_ids)
valid_test_overlap = valid_company_ids.intersection(test_ids)

train_dates = model_df.loc[train_idx, DATE_COL]
valid_dates = model_df.loc[valid_idx, DATE_COL]
test_dates = model_df.loc[test_idx, DATE_COL]

print(f"Development rows before validation split: {development_mask.sum():,}")
print(f"Model train rows: {len(X_train):,}")
print(f"Validation rows: {len(X_valid):,}")
print(f"Final test rows: {len(X_test):,}")
print(f"Model train companies: {len(train_company_ids):,}")
print(f"Validation companies: {len(valid_company_ids):,}")
print(f"Final test companies: {len(test_ids):,}")
print(f"Companies in both model train and validation: {len(train_valid_overlap):,}")
print(f"Model train companies also in final test: {len(train_test_overlap):,}")
print(f"Validation companies also in final test: {len(valid_test_overlap):,}")
if SPLIT_STRATEGY == "out_of_time":
    print("Final-test company overlap is expected because the final test is split by future period.")
elif SPLIT_STRATEGY == "out_of_sample":
    if len(train_test_overlap) != 0 or len(valid_test_overlap) != 0:
        raise ValueError(
            "Out-of-sample final test should have zero company overlap with model train and validation."
        )
    print("Final-test company overlap is 0, as expected for the out-of-sample split.")
else:
    raise ValueError(f"Unexpected SPLIT_STRATEGY: {SPLIT_STRATEGY}")
print(f"Model train date range: {train_dates.min().date()} to {train_dates.max().date()}")
print(f"Validation date range: {valid_dates.min().date()} to {valid_dates.max().date()}")
print(f"Final test date range: {test_dates.min().date()} to {test_dates.max().date()}")
print(f"Model train default rate: {y_train.mean() * 100:.4f}%")
print(f"Validation default rate: {y_valid.mean() * 100:.4f}%")
print(f"Final test default rate: {y_test.mean() * 100:.4f}%")


Development rows before validation split: 119,128
Model train rows: 95,179
Validation rows: 23,949
Final test rows: 6,630
Model train companies: 36,645
Validation companies: 9,161
Final test companies: 6,606
Companies in both model train and validation: 0
Model train companies also in final test: 5,035
Validation companies also in final test: 1,308
Final-test company overlap is expected because the final test is split by future period.
Model train date range: 2015-01-01 to 2020-12-31
Validation date range: 2015-01-30 to 2020-12-31
Final test date range: 2021-01-01 to 2021-12-31
Model train default rate: 7.0856%
Validation default rate: 7.0400%
Final test default rate: 4.8115%


## 3. Fit logistic regression

The model is fitted on the training sample using the selected WoE features. The model is intentionally simple: logistic regression without regularization, so the coefficients remain easy to inspect.


In [4]:
logit_model = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=1000,
)

logit_model.fit(X_train, y_train)

coef_table = pd.DataFrame(
    {
        "feature": selected_features,
        "feature_name": [feature_names.get(feature, feature) for feature in selected_features],
        "coefficient": logit_model.coef_[0],
    }
)
coef_table["abs_coefficient"] = coef_table["coefficient"].abs()
coef_table = coef_table.sort_values("abs_coefficient", ascending=False)

print(f"Intercept: {logit_model.intercept_[0]:.6f}")
display(coef_table)


Intercept: -2.593105


,feature,feature_name,coefficient,abs_coefficient
2,Var_04,Current Ratio,-3.575074,3.575074
4,Var_35,Total Liabilities Total Assets,-1.746373,1.746373
0,Var_02,Assets Total,-1.365746,1.365746
8,Var_07,EBITDA Margin,-0.966643,0.966643
5,Var_23,Net Profit Margin,-0.346287,0.346287
6,Var_33,Senior Net Debt EBITDA,-0.343714,0.343714
3,Var_15,Gross Profit Margin,0.237994,0.237994
7,Var_06_missing_flag,Depreciation And Impairment (missing flag),-0.232955,0.232955
1,Var_28,Return On Total Equity Reserve,0.168930,0.168930


## 4. Statistical significance of variables

`sklearn` is used for fitting and prediction, but it does not report p-values. Therefore, the same training data is also fitted with `statsmodels.Logit` to check coefficient significance. A variable is treated as statistically significant when its p-value is below 0.05.


In [5]:
X_train_sm = sm.add_constant(X_train)

statsmodels_logit = sm.Logit(y_train, X_train_sm)
statsmodels_result = statsmodels_logit.fit(disp=False)

significance_table = pd.DataFrame(
    {
        "feature": statsmodels_result.params.index,
        "coefficient": statsmodels_result.params.values,
        "std_error": statsmodels_result.bse.values,
        "z_stat": statsmodels_result.tvalues.values,
        "p_value": statsmodels_result.pvalues.values,
    }
)

significance_table = significance_table[significance_table["feature"] != "const"].copy()
significance_table["feature_name"] = significance_table["feature"].map(feature_names)
significance_table["significant_5pct"] = significance_table["p_value"] < 0.05
significance_table = significance_table[
    ["feature", "feature_name", "coefficient", "std_error", "z_stat", "p_value", "significant_5pct"]
]
significance_table = significance_table.sort_values("p_value")

print(f"Statistically significant features at 5%: {significance_table['significant_5pct'].sum()} / {len(significance_table)}")
display(significance_table)


Statistically significant features at 5%: 9 / 9


,feature,feature_name,coefficient,std_error,z_stat,p_value,significant_5pct
1,Var_02,Assets Total,-1.367416,0.020734,-65.950798,0.000000e+00,True
3,Var_04,Current Ratio,-3.571068,0.059732,-59.784397,0.000000e+00,True
5,Var_35,Total Liabilities Total Assets,-1.742800,0.079914,-21.808477,1.927870e-105,True
9,Var_07,EBITDA Margin,-0.998733,0.092508,-10.796185,3.588020e-27,True
6,Var_23,Net Profit Margin,-0.338748,0.072258,-4.688054,2.758159e-06,True
7,Var_33,Senior Net Debt EBITDA,-0.336828,0.075991,-4.432486,9.315260e-06,True
2,Var_28,Return On Total Equity Reserve,0.167328,0.039185,4.270245,1.952579e-05,True
4,Var_15,Gross Profit Margin,0.219191,0.052697,4.159458,3.190037e-05,True
8,Var_06_missing_flag,Depreciation And Impairment (missing flag),-0.228112,0.074198,-3.074368,2.109493e-03,True


## 5. Sanity-check coefficient signs

This section checks whether each coefficient sign is consistent with the observed WoE direction. For a default model, if higher WoE is more common among non-defaults, the expected coefficient sign is usually negative.


In [6]:
economic_intuition = {
    "Var_02": "Higher assets usually mean larger, more stable companies.",
    "Var_28": "Higher return on equity/reserves usually means better profitability.",
    "Var_04": "Higher current ratio usually means better liquidity.",
    "Var_15": "Higher gross profit margin usually means better profitability.",
    "Var_35": "Higher liabilities/assets usually means higher leverage risk.",
    "Var_23": "Higher net profit margin usually means better profitability.",
    "Var_33": "Higher senior net debt to EBITDA usually means higher leverage risk.",
    "Var_05": "Higher debt/net worth usually means higher leverage risk.",
    "Var_06_missing_flag": "Missing depreciation data may contain risk information.",
    "Var_07": "Higher EBITDA margin usually means better operating profitability.",
}

sign_rows = []

for feature, coefficient in zip(selected_features, logit_model.coef_[0]):
    mean_woe_non_default = X_train.loc[y_train == 0, feature].mean()
    mean_woe_default = X_train.loc[y_train == 1, feature].mean()

    if mean_woe_non_default > mean_woe_default:
        expected_sign = "negative"
        woe_direction = "higher WoE observed for non-defaults"
    elif mean_woe_non_default < mean_woe_default:
        expected_sign = "positive"
        woe_direction = "higher WoE observed for defaults"
    else:
        expected_sign = "near zero"
        woe_direction = "same mean WoE for both classes"

    if coefficient < 0:
        actual_sign = "negative"
        model_effect = "higher WoE -> lower predicted default risk"
    elif coefficient > 0:
        actual_sign = "positive"
        model_effect = "higher WoE -> higher predicted default risk"
    else:
        actual_sign = "zero"
        model_effect = "no effect"

    sign_rows.append(
        {
            "feature": feature,
            "feature_name": feature_names.get(feature, feature),
            "coefficient": coefficient,
            "actual_sign": actual_sign,
            "expected_sign_from_woe": expected_sign,
            "sign_check": "OK" if actual_sign == expected_sign else "CHECK",
            "mean_woe_non_default": mean_woe_non_default,
            "mean_woe_default": mean_woe_default,
            "woe_direction": woe_direction,
            "model_effect": model_effect,
            "economic_intuition": economic_intuition.get(feature, ""),
        }
    )

sign_check_table = pd.DataFrame(sign_rows).sort_values(
    ["sign_check", "coefficient"],
    ascending=[True, True],
)

features_to_review = sign_check_table[sign_check_table["sign_check"] == "CHECK"]

print(f"Features with signs to review: {len(features_to_review)}")
display(sign_check_table)


Features with signs to review: 2


,feature,feature_name,coefficient,actual_sign,expected_sign_from_woe,sign_check,mean_woe_non_default,mean_woe_default,woe_direction,model_effect,economic_intuition
1,Var_28,Return On Total Equity Reserve,0.168930,positive,negative,CHECK,0.147054,-0.125261,higher WoE observed for non-defaults,higher WoE -> higher predicted default risk,Higher return on equity/reserves usually means...
3,Var_15,Gross Profit Margin,0.237994,positive,negative,CHECK,0.056390,-0.055261,higher WoE observed for non-defaults,higher WoE -> higher predicted default risk,Higher gross profit margin usually means bette...
2,Var_04,Current Ratio,-3.575074,negative,negative,OK,0.057897,-0.057968,higher WoE observed for non-defaults,higher WoE -> lower predicted default risk,Higher current ratio usually means better liqu...
4,Var_35,Total Liabilities Total Assets,-1.746373,negative,negative,OK,0.031490,-0.032408,higher WoE observed for non-defaults,higher WoE -> lower predicted default risk,Higher liabilities/assets usually means higher...
0,Var_02,Assets Total,-1.365746,negative,negative,OK,3.267030,-1.507923,higher WoE observed for non-defaults,higher WoE -> lower predicted default risk,"Higher assets usually mean larger, more stable..."
8,Var_07,EBITDA Margin,-0.966643,negative,negative,OK,0.019632,-0.021913,higher WoE observed for non-defaults,higher WoE -> lower predicted default risk,Higher EBITDA margin usually means better oper...
5,Var_23,Net Profit Margin,-0.346287,negative,negative,OK,0.029281,-0.033556,higher WoE observed for non-defaults,higher WoE -> lower predicted default risk,Higher net profit margin usually means better ...
6,Var_33,Senior Net Debt EBITDA,-0.343714,negative,negative,OK,0.029259,-0.033234,higher WoE observed for non-defaults,higher WoE -> lower predicted default risk,Higher senior net debt to EBITDA usually means...
7,Var_06_missing_flag,Depreciation And Impairment (missing flag),-0.232955,negative,negative,OK,0.020067,-0.025516,higher WoE observed for non-defaults,higher WoE -> lower predicted default risk,Missing depreciation data may contain risk inf...


## 6. Review flagged coefficient signs

Flagged variables are checked with one-feature logistic regressions. This helps distinguish variables that behave unusually on their own from variables whose sign changes only after being combined with other predictors.


In [7]:
review_rows = []

for feature in features_to_review["feature"]:
    one_feature_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=1000,
    )
    one_feature_model.fit(X_train[[feature]], y_train)

    full_coefficient = float(
        coef_table.loc[coef_table["feature"] == feature, "coefficient"].iloc[0]
    )
    one_feature_coefficient = float(one_feature_model.coef_[0][0])

    full_sign = "positive" if full_coefficient > 0 else "negative" if full_coefficient < 0 else "zero"
    one_feature_sign = (
        "positive"
        if one_feature_coefficient > 0
        else "negative"
        if one_feature_coefficient < 0
        else "zero"
    )

    mean_woe_non_default = X_train.loc[y_train == 0, feature].mean()
    mean_woe_default = X_train.loc[y_train == 1, feature].mean()

    review_rows.append(
        {
            "feature": feature,
            "feature_name": feature_names.get(feature, feature),
            "full_model_coefficient": full_coefficient,
            "full_model_sign": full_sign,
            "one_feature_coefficient": one_feature_coefficient,
            "one_feature_sign": one_feature_sign,
            "sign_flip_in_full_model": "YES" if full_sign != one_feature_sign else "NO",
            "mean_woe_non_default": mean_woe_non_default,
            "mean_woe_default": mean_woe_default,
        }
    )

sign_review_table = pd.DataFrame(review_rows)

if sign_review_table.empty:
    print("No flagged coefficient signs to review.")
else:
    display(sign_review_table)


,feature,feature_name,full_model_coefficient,full_model_sign,one_feature_coefficient,one_feature_sign,sign_flip_in_full_model,mean_woe_non_default,mean_woe_default
0,Var_28,Return On Total Equity Reserve,0.168930,positive,-1.017618,negative,YES,0.147054,-0.125261
1,Var_15,Gross Profit Margin,0.237994,positive,-1.024899,negative,YES,0.056390,-0.055261


## 7. Evaluate on train, validation, and test

AUC and Gini are calculated from predicted default probabilities. The classification report is shown at threshold 0.50 for train, validation, and test.

The validation results are used for model checking and threshold selection. The 2021 test set remains untouched until the final evaluation, so it represents the independent out-of-time result. A lower test AUC than train/validation AUC is expected because the test period is a later economic period.


In [8]:
train_default_probability = logit_model.predict_proba(X_train)[:, 1]
valid_default_probability = logit_model.predict_proba(X_valid)[:, 1]
test_default_probability = logit_model.predict_proba(X_test)[:, 1]

train_prediction = (train_default_probability >= 0.5).astype(int)
valid_prediction = (valid_default_probability >= 0.5).astype(int)
test_prediction = (test_default_probability >= 0.5).astype(int)

train_auc = roc_auc_score(y_train, train_default_probability)
train_gini = 2 * train_auc - 1
valid_auc = roc_auc_score(y_valid, valid_default_probability)
valid_gini = 2 * valid_auc - 1
test_auc = roc_auc_score(y_test, test_default_probability)
test_gini = 2 * test_auc - 1

metrics_table = pd.DataFrame(
    {
        "metric": ["Train AUC", "Train Gini", "Validation AUC", "Validation Gini", "Test AUC", "Test Gini"],
        "value": [train_auc, train_gini, valid_auc, valid_gini, test_auc, test_gini],
    }
)

display(metrics_table)

print("Train classification report at threshold 0.50")
print(classification_report(y_train, train_prediction, digits=4))
print("Validation classification report at threshold 0.50")
print(classification_report(y_valid, valid_prediction, digits=4))
print("Test classification report at threshold 0.50")
print(classification_report(y_test, test_prediction, digits=4))


,metric,value
0,Train AUC,0.954168
1,Train Gini,0.908336
2,Validation AUC,0.953171
3,Validation Gini,0.906343
4,Test AUC,0.920148
5,Test Gini,0.840297


Train classification report at threshold 0.50
              precision    recall  f1-score   support

           0     0.9609    0.9873    0.9739     88435
           1     0.7393    0.4727    0.5767      6744

    accuracy                         0.9508     95179
   macro avg     0.8501    0.7300    0.7753     95179
weighted avg     0.9452    0.9508    0.9458     95179

Validation classification report at threshold 0.50
              precision    recall  f1-score   support

           0     0.9620    0.9865    0.9741     22263
           1     0.7319    0.4858    0.5840      1686

    accuracy                         0.9513     23949
   macro avg     0.8470    0.7361    0.7790     23949
weighted avg     0.9458    0.9513    0.9467     23949

Test classification report at threshold 0.50
              precision    recall  f1-score   support

           0     0.9673    0.9905    0.9788      6311
           1     0.6429    0.3386    0.4435       319

    accuracy                         0.9

## 8. Threshold sweep on validation

The model outputs probabilities, so a threshold is needed to convert probabilities into default / non-default predictions. The threshold is selected on the internal validation set by maximizing F1 for the default class. The selected threshold is then applied once to the independent out-of-time test set.

The selected threshold should not be interpreted as guaranteed to improve test F1. Its purpose is to choose a business operating point using validation data only. On the final test set, the threshold may trade higher recall for lower precision, so the final choice should depend on the cost of missed defaults versus false alarms.


In [9]:
threshold_rows = []

for threshold in [i / 100 for i in range(1, 100)]:
    threshold_prediction = (valid_default_probability >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_valid,
        threshold_prediction,
        average="binary",
        zero_division=0,
    )

    threshold_rows.append(
        {
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "predicted_default_rate": threshold_prediction.mean(),
            "predicted_defaults": int(threshold_prediction.sum()),
        }
    )

threshold_table = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_table.sort_values("f1", ascending=False).iloc[0]
best_threshold = best_threshold_row["threshold"]
test_prediction_best_threshold = (test_default_probability >= best_threshold).astype(int)

print(f"Best threshold by validation F1: {best_threshold:.2f}")
print(f"Best validation F1: {best_threshold_row['f1']:.4f}")
print("Test classification report at selected threshold")
print(classification_report(y_test, test_prediction_best_threshold, digits=4))

display(threshold_table.sort_values("f1", ascending=False).head(10))


Best threshold by validation F1: 0.34
Best validation F1: 0.6255
Test classification report at selected threshold
              precision    recall  f1-score   support

           0     0.9716    0.9716    0.9716      6311
           1     0.4389    0.4389    0.4389       319

    accuracy                         0.9460      6630
   macro avg     0.7053    0.7053    0.7053      6630
weighted avg     0.9460    0.9460    0.9460      6630



,threshold,precision,recall,f1,predicted_default_rate,predicted_defaults
33,0.34,0.589501,0.666074,0.625453,0.079544,1905
29,0.30,0.553924,0.715896,0.624580,0.090985,2179
32,0.33,0.580530,0.675563,0.624452,0.081924,1962
27,0.28,0.537999,0.743179,0.624159,0.097248,2329
34,0.35,0.597607,0.651839,0.623546,0.076788,1839
30,0.31,0.561370,0.699881,0.623020,0.087770,2102
31,0.32,0.569533,0.687426,0.622951,0.084972,2035
28,0.29,0.544242,0.725979,0.622109,0.093908,2249
35,0.36,0.602018,0.637011,0.619020,0.074492,1784
26,0.27,0.525156,0.749110,0.617453,0.100422,2405


## 9. Simple scorecard from coefficients

The scorecard converts logistic regression log-odds into points. The configuration uses base score 600 and PDO 50. Higher score means lower predicted default risk. The score distribution is checked on the test set.


In [10]:
BASE_SCORE = 600
PDO = 50
FACTOR = PDO / log(2)

intercept_points = BASE_SCORE - FACTOR * logit_model.intercept_[0]

scorecard_table = pd.DataFrame(
    {
        "feature": selected_features,
        "feature_name": [feature_names.get(feature, feature) for feature in selected_features],
        "coefficient": logit_model.coef_[0],
    }
)
scorecard_table["points_per_woe_unit"] = -FACTOR * scorecard_table["coefficient"]
scorecard_table = scorecard_table.sort_values("points_per_woe_unit", ascending=False)

print(f"Base score: {BASE_SCORE}")
print(f"PDO: {PDO}")
print(f"Factor: {FACTOR:.4f}")
print(f"Intercept points: {intercept_points:.2f}")

display(scorecard_table)


Base score: 600
PDO: 50
Factor: 72.1348
Intercept points: 787.05


,feature,feature_name,coefficient,points_per_woe_unit
2,Var_04,Current Ratio,-3.575074,257.887095
4,Var_35,Total Liabilities Total Assets,-1.746373,125.974168
0,Var_02,Assets Total,-1.365746,98.517716
8,Var_07,EBITDA Margin,-0.966643,69.728576
5,Var_23,Net Profit Margin,-0.346287,24.979332
6,Var_33,Senior Net Debt EBITDA,-0.343714,24.793711
7,Var_06_missing_flag,Depreciation And Impairment (missing flag),-0.232955,16.804159
1,Var_28,Return On Total Equity Reserve,0.168930,-12.185707
3,Var_15,Gross Profit Margin,0.237994,-17.167609


In [11]:
test_log_odds = logit_model.decision_function(X_test)
test_score = BASE_SCORE - FACTOR * test_log_odds

test_scores = model_df.loc[X_test.index, [ID_COL, DATE_COL, TARGET_COL]].copy()
test_scores["predicted_default_probability"] = test_default_probability
test_scores["score"] = test_score

score_summary = (
    test_scores
    .groupby(TARGET_COL)["score"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)

display(score_summary)
display(test_scores.head())


,default,count,mean,median,min,max
0,0,6311,1097.378162,1163.526514,483.092742,1573.739579
1,1,319,672.851049,661.388168,349.041935,1139.767265


,ID,obs_date,default,predicted_default_probability,score
43,20531,2021-12-31,0,0.000060,1301.448683
47,21521,2021-12-31,0,0.000328,1178.746808
54,24255,2021-06-30,0,0.001553,1066.405641
104,64671,2021-09-30,0,0.000010,1433.700454
105,64671,2021-12-31,0,0.000010,1429.416317


## Final report

The notebook trains a WoE logistic regression model using **9 selected WoE features**. In the current run, it reads `data/final/logit_selected_woe_dataset_out_of_time.csv`, so 2021 remains the independent out-of-time test period and the 2015-2020 development period is split into model train and validation using stratified company-level sampling.

### Split design

The validation design is hybrid. The final test is based on time, while the internal validation split is based on companies:

- development period before validation split: **119,128 rows**
- model train: **95,179 rows**, **36,645 companies**, dates from **2015-01-01** to **2020-12-31**
- validation: **23,949 rows**, **9,161 companies**, dates from **2015-01-30** to **2020-12-31**
- final test: **6,630 rows**, **6,606 companies**, dates from **2021-01-01** to **2021-12-31**
- companies in both model train and validation: **0**
- company overlap with final test is expected because final test is split by future period

The final test answers the main forecasting question: whether a model developed on earlier observations still performs in a later period. The company-level internal validation split is an additional safeguard because it prevents the same company from appearing in both model train and validation.

### Leakage discussion

The champion notebook itself does not fit the model or select the threshold on the test set. Logistic regression is fitted only on the model train sample, the classification threshold is selected only on validation, and the final 2021 test set is used only for final evaluation.

### Model performance

Model ranking performance is strong on train and validation, and remains good on the out-of-time test set:

- train AUC: **0.9542**
- train Gini: **0.9083**
- validation AUC: **0.9532**
- validation Gini: **0.9063**
- test AUC: **0.9201**
- test Gini: **0.8403**

The test performance is lower than train and validation performance, which is expected for an out-of-time test. 

All **9 out of 9** model variables are statistically significant at the 5% level according to `statsmodels.Logit`.

### Threshold interpretation

At threshold **0.50**, test default-class performance is:

- precision: **0.6429**
- recall: **0.3386**
- F1-score: **0.4435**

The threshold sweep is done on validation. It selected threshold **0.34**, with validation F1 equal to **0.6255**. Applying this threshold to the independent 2021 test set gives:

- precision: **0.4389**
- recall: **0.4389**
- F1-score: **0.4389**

The validation-selected threshold increases recall on the test set compared with threshold 0.50, but precision decreases and the final test F1 is slightly lower. Therefore, threshold tuning should not be described as improving test F1. It should be interpreted as a business trade-off between missing defaults and producing false alarms.

### OOT vs OOS test F1

For comparison, the same notebook can also be run with `SPLIT_STRATEGY = "out_of_sample"`. The resulting test F1 values are:

- OOT test F1 at threshold **0.50**: **0.4435**
- OOT test F1 at validation-selected threshold **0.34**: **0.4389**
- OOS test F1 at threshold **0.50**: **0.5578**
- OOS test F1 at validation-selected threshold **0.31**: **0.6092**

The out-of-sample split gives higher test F1 than the out-of-time split, which is reasonable because OOS tests generalization to unseen companies, while OOT tests generalization to a later economic period.

### Coefficient interpretation

The coefficient sign check flagged two variables for review:

- Return On Total Equity Reserve
- Gross Profit Margin

For these variables, the one-feature model has the expected sign, but the full multivariate model flips the sign. This suggests overlap or correlation with other selected predictors. These variables remain statistically significant, but their economic interpretation should be treated carefully.

### Scorecard interpretation

The simple scorecard uses base score **600** and PDO **50**. On the test set, the score separates defaults and non-defaults clearly:

- mean score for non-defaults: **1097.38**
- mean score for defaults: **672.85**

Overall, this notebook produces a complete champion model workflow for the current `out_of_time` final dataset: it keeps 2021 as an independent test set, creates an internal stratified company-level validation split inside 2015-2020, fits the logistic regression, checks statistical and economic behavior, selects a threshold without touching the test set, evaluates final performance on the 2021 test period, and converts the model into a simple scorecard.
